In [ ]:
import numpy as np
import os
import torch
from PIL import Image
import json

from monai.networks.nets import UNet, AttentionUnet
from src.read_data_transform_fun import to_0_1_format_img, to_0_255_format_img
from monai.networks.layers import Norm

import random

from src.pipeliner import Pipeliner
from src.lr_scheduler import lr_scheduler_200, lr_scheduler_100

import datetime
import re

from read_fun import sort_filenames, check_sorted
from generator3D import get_device_tensor_from_list_of_numpy, data_generator, PiplinerPaperGen, added_zero_channel_axis, get_cpu_numpy_from_list_of_tensor, get_device_tensor_from_channel_stack_of_numpy

from split_and_glue import split_3d_data, glue_data
from predict import data_list_to_predict_gen, predict
from save_data import saveDataframeAsImgs
from test_metrics import Dice, Jaccard

In [ ]:
DATASET_PATH_DATA = "D:/Data/datasets/Lucchi++/Train_In"
DATASET_PATH_MASK = "D:/Data/datasets/Lucchi++/Train_Out"

# хороший датасет dataset_2026_07_12__16_00_02

# dataset_2026_07_15__15_32_05
SYN_DATASET_PATH_DATA = "D:/Projects/Synthetics/Synthetic3D/datasets/dataset_2026_07_12__16_00_02/original"
SYN_DATASET_PATH_MASK = "D:/Projects/Synthetics/Synthetic3D/datasets/dataset_2026_07_12__16_00_02"

number_of_classes = 5
classes_list = ["mitohondrion", "membranes", "vesicles", "axon", "psd"]

AVAILABLE_IMG_TYPE = ('.png', '.jpg', '.jpeg')

shape_of_data = (80, 80, 80)    # быстрее работает
#shape_of_data = (96, 96, 96)    # быстрее работает
#hape_of_data = (128, 128, 128) # долго тренится но больше охват

## **ЧТЕНИЕ ДАННЫХ**

### Чтение основного датасета Lucchi++

In [ ]:
img_names_all = [name for name in os.listdir(DATASET_PATH_DATA) if name.endswith((AVAILABLE_IMG_TYPE))]
print(check_sorted(img_names_all))
print(img_names_all)

print("\nполучение данных из слоев")
# получение данных из слоев
list_of_images = [Image.open(os.path.join(DATASET_PATH_DATA, name)) for name in img_names_all]
dataframe = np.stack(list_of_images)
print(dataframe.shape)
dataframe = to_0_1_format_img(dataframe)
print(dataframe.min(), dataframe.max())

print("\nполучение данных из масок ")
# получение данных из масок 
list_of_masks = [Image.open(os.path.join(DATASET_PATH_MASK, name)) for name in img_names_all]
mitoframe = np.stack(list_of_masks)
print(mitoframe.shape)
mitoframe = to_0_1_format_img(mitoframe)
print(mitoframe.min(), mitoframe.max())

print()


### Чтение Синтетики

In [ ]:
syn_img_names_all = [name for name in os.listdir(SYN_DATASET_PATH_DATA) if name.endswith((AVAILABLE_IMG_TYPE))]

syn_img_names_all = sorted(syn_img_names_all, key=len)

print(check_sorted(syn_img_names_all))
print(syn_img_names_all)

for i in range(256):
    if f'synthetic_data_512_512_{i}.png' != syn_img_names_all[i]:
        print("ERROR!!")

print("\nполучение данных из слоев")
# получение данных из слоев
syn_list_of_images = [Image.open(os.path.join(SYN_DATASET_PATH_DATA, name)) for name in syn_img_names_all]
syn_dataframe = np.stack(syn_list_of_images)
print(syn_dataframe.shape)
syn_dataframe = to_0_1_format_img(syn_dataframe)
print(syn_dataframe.min(), syn_dataframe.max())

print("\nполучение данных из масок ")
# получение данных из масок
syn_maskframe = []
for i in range(number_of_classes):
    class_name = classes_list[i]
    syn_list_of_masks = [Image.open(os.path.join(SYN_DATASET_PATH_MASK, class_name, name)) for name in syn_img_names_all]
    syn_mask = np.stack(syn_list_of_masks)
    print(syn_mask.shape)
    syn_mask = to_0_1_format_img(syn_mask)
    print(syn_mask.min(), syn_mask.max())
    syn_maskframe.append(syn_mask)

syn_maskframe = np.stack(syn_maskframe)
print(syn_maskframe.shape)

### Выбор девайса и перенос данных на устройство

In [ ]:
if torch.cuda.is_available():
   device = torch.device("cuda")
else:
    raise Exception("Augmentator don't use GPU device !")

In [ ]:
dataTensor = get_device_tensor_from_list_of_numpy(added_zero_channel_axis(dataframe), device)
mitoTensor = get_device_tensor_from_list_of_numpy(added_zero_channel_axis(mitoframe), device)

print(type(dataTensor), dataTensor.shape)
print(type(mitoTensor), mitoTensor.shape)


In [ ]:
syn_dataTensor = get_device_tensor_from_list_of_numpy(added_zero_channel_axis(syn_dataframe), device)
syn_maskTensor = get_device_tensor_from_channel_stack_of_numpy(syn_maskframe, device)

print(type(syn_dataTensor), syn_dataTensor.shape)
print(type(syn_maskTensor), syn_maskTensor.shape)

### Тест генератора

In [ ]:
'''
from monai.visualize import plot_2d_or_3d_image
from monai.visualize import matshow3d
import matplotlib.pyplot as plt

augmentation = {
                "rotation_range": 3 * 0.01745, # перевод в радианы
                #"width_shift_range": 0.05,
                #"height_shift_range": 0.05,
                #"depth_shift_range": 0.05,
                "zoom_range": 0.1,
                "flip": True,
                "noise_limit": 3,
                "p_rotate_90": 0.5,
                "fill_mode": "zeros"
                }

test_train_generator = data_generator(#[[dataTensor, mitoTensor],
                                      # [syn_dataTensor, syn_maskTensor]
                                      #],
                                      [syn_dataTensor, syn_maskTensor],
                                      (80, 80, 80),
                                      3,
                                      5,
                                      augmentation,
                                      device=device,
                                      #probability_choice_gen=[0.75, 0.25]
                                      )
print(f"gen_create, {len(test_train_generator)}")


for i in range(len(test_train_generator)):
    x, y = test_train_generator[i]
    print(x.shape, y.shape)

    # Преобразуем в numpy
    for i in range(x.shape[0]):
        data_array_3d = x[i].detach().cpu().numpy()
        mask_array_3d = y[i].detach().cpu().numpy()
        print(mask_array_3d.max())

        for i in range(number_of_classes):
            fig = plt.figure()
            matshow3d([data_array_3d[0], mask_array_3d[i,:,:,:]], title="data and mask")
            plt.show()
'''


## Подготовка модели

### инициализация модели

изначальные параметры
model = UNet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    act="RELU",
    num_res_units=0,
    dropout=0.5,
).to(device)


In [ ]:
'''
modelName = f"3D_UNet_dropaut_0_num_class_{number_of_classes}"
model = UNet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    act="RELU",
    num_res_units=0,
    dropout=0.25,
).to(device)
'''

'''
modelName = f"3D_Attention_UNet_dropaut_0_num_class_{number_of_classes}"
# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@ base
model = AttentionUnet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=number_of_classes,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    dropout=0.25,
).to(device)
'''



'''
modelName = f"3D_tiny_Attention_UNet_dropaut_0_num_class_{number_of_classes}"
model = AttentionUnet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=number_of_classes,
    channels=(32, 32, 64, 64, 64),
    strides=(2, 2, 2, 2),
    dropout=0.25,
).to(device)
'''


'''
modelName = f"3D_Residual_UNet_dropaut_0_num_class_{number_of_classes}"
model = UNet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=number_of_classes,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    act="RELU",
    num_res_units=1,
    dropout=0.25,
).to(device)
'''


modelName = f"3D_tiny_Resudial_UNet_dropaut_0_num_class_{number_of_classes}"
model = UNet(
    spatial_dims=3, # 3D
    in_channels=1,
    out_channels=number_of_classes,
    channels=(32, 32, 64, 64, 64),
    strides=(2, 2, 2, 2),
    act="RELU",
    num_res_units=1,
    dropout=0.25,
    norm = Norm.BATCH
).to(device)






### Подготовка тренировщика
pipliner = Pipeliner(model = model,
                     last_activation_name = "sigmoid_activation",
                     num_classes = number_of_classes,
                     num_channel = 1,
                     device = device,
                     silence_mode = False,
                     task_mode = "segmentation")


## Трениров

### сборка генератора для тренировки

In [ ]:
augmentation = {
                #"rotation_range": 3 * 0.01745, # перевод в радианы
                #"width_shift_range": 0.05,
                #"height_shift_range": 0.05,
                #"depth_shift_range": 0.05,
                #"zoom_range": 0.1,
                "flip": True,
                "noise_limit": 2,
                "p_rotate_90": 0.5,
                "fill_mode": "zeros"
                }

# start param batch_size = 6
batch_size = 6

test_train_generator = data_generator(
                                      #datasets = [dataTensor, mitoTensor],                        # только реальные
                                      #datasets = [[dataTensor, mitoTensor],                      # микс
                                      #            [syn_dataTensor, syn_maskTensor]],             # микс
                                      #probability_choice_gen=[0.8, 0.2], # 20% синтетики         # микс
                                      datasets = [syn_dataTensor, syn_maskTensor],               # только синтетика
                                      target_shape = shape_of_data,
                                      batch_size = batch_size,
                                      count_of_exsamples = 500,
                                      aug_dict = augmentation,
                                      is_augment = True,
                                      device=device)


#modelName_with_gen_data = modelName + f"_orig_only_batch_6_shape_data_{shape_of_data[0]}"
#modelName_with_gen_data = modelName + f"_mix__only_batch_6_shape_data_{shape_of_data[0]}"
modelName_with_gen_data = modelName + f"_sint_only_batch_6_shape_data_{shape_of_data[0]}"
pipliner_gen = PiplinerPaperGen(test_train_generator, list_class_name = classes_list)

### запуск тренировки

In [ ]:
now = datetime.datetime.now()
data_save = f"_{now.year:04}_{now.month:02}_{now.day:02}_{now.hour:02}_{now.minute:02}_{now.second:02}"

UsemodelName = modelName_with_gen_data + data_save

history = pipliner.train(pipliner_gen,
                         num_epoch = 150,
                         optimizer_name = "Adam",
                         metric_names = ["Dice",
                                         "DiceMultilabel",
                                         #MultiMetricClasses"
                                        ],
                         losses = ["DiceLossMulticlass"],
                         lr_scheduler = lr_scheduler_200,
                         use_validation = False,
                         use_train_metric = True,
                         train_args = {},
                         model_name= UsemodelName,
                         device=device,
                         save_model_mode = "weights_only_after_finish",  # [ "all", "weights_only", "weights_only_after_finish"]
                         save_pipeliner_mode = "save_pipeliner"  # ["no_save", "save_pipeliner"]
                        )

In [ ]:
'''
now = datetime.datetime.now()
data_save = f"_{now.year:04}_{now.month:02}_{now.day:02}_{now.hour:02}_{now.minute:02}_{now.second:02}"

UsemodelName = modelName_with_gen_data + data_save

history = pipliner.train(pipliner_gen,
                         num_epoch = 2,
                         optimizer_name = "Adam",
                         metric_names = ["Dice",
                                         "DiceMultilabel",
                                         #MultiMetricClasses"
                                        ],
                         losses = ["DiceLossMulticlass"],
                         lr_scheduler = lr_scheduler_200,
                         use_validation = False,
                         use_train_metric = True,
                         train_args = {},
                         model_name= UsemodelName,
                         device=device,
                         save_model_mode = "weights_only_after_finish",  # [ "all", "weights_only", "weights_only_after_finish"]
                         save_pipeliner_mode = "save_pipeliner"  # ["no_save", "save_pipeliner"]
                        )
'''

In [ ]:
try:
    history['lr'] = np.array(history['lr']).astype(float).tolist()
except:
    print("WARNING: no lr")

with open("history_" + UsemodelName + '.json', 'w') as file:
    json.dump(history, file, indent=4)

## Тестирование

### Загрузка тестового датасета

In [ ]:
TEST_DATASET_PATH_DATA = "D:/Data/datasets/Lucchi++/Test_In"
TEST_DATASET_PATH_MASK = "D:/Data/datasets/Lucchi++/Test_Out"

test_img_names_all = [name for name in os.listdir(TEST_DATASET_PATH_DATA) if name.endswith((AVAILABLE_IMG_TYPE))]
print(test_img_names_all)

print("\ntest_dataframe")
list_of_test_images = [Image.open(os.path.join(TEST_DATASET_PATH_DATA, name)) for name in test_img_names_all]
test_dataframe = np.stack(list_of_test_images)
print(test_dataframe.shape)
test_dataframe = to_0_1_format_img(test_dataframe)
print(test_dataframe.min(), test_dataframe.max())

print("\ntest_masks")
list_of_test_masks = [Image.open(os.path.join(TEST_DATASET_PATH_MASK, name)) for name in test_img_names_all]

for i, mask in enumerate(list_of_test_masks):
    list_of_test_masks[i] = mask.convert('L')

test_mitoframe = np.stack(list_of_test_masks)
print(test_mitoframe.shape)
test_mitoframe = to_0_1_format_img(test_mitoframe)
print(test_mitoframe.min(), test_mitoframe.max())

test_dataTensor = get_device_tensor_from_list_of_numpy(added_zero_channel_axis(test_dataframe), device)

print(type(test_dataTensor), test_dataTensor.shape)


### Split data

In [ ]:
# test split_3d_data
'''
import numpy as np
import matplotlib.pyplot as plt
data = np.zeros((1, 256, 257, 383), dtype=np.uint8)
for s_y in [0, 20, 40, 80, 120, 200]:
    for s_x in [50, 100, 150, 250]:
        data[:,:, s_y:s_y+10, s_x:s_x+5] = 255

# Функция отображения изображения
def show_slice(slice_2d):
    plt.imshow(slice_2d, cmap='gray')
    plt.axis('off')
    plt.show()

show_slice(data[0,0,:,:])

frame_list, split_data = split_3d_data(data, (256, 256, 256), 64)

print(split_data)

for frame in frame_list:
    show_slice(frame[0,0,:,:])
'''

### Run predict

In [ ]:
test_frames, split_data = split_3d_data(test_dataTensor, shape_of_data, shape_of_data[0]-40)
result = predict(pipliner, data_list_to_predict_gen(test_frames), device)

In [ ]:
print (result[0].shape)

### glue_data

#### glue test

In [ ]:
test_tiles, sp_data = split_3d_data(test_dataTensor, shape_of_data, shape_of_data[0]-40)
ret_split_test_data = glue_data(test_tiles, sp_data, shape_of_data)

print ((ret_split_test_data-get_cpu_numpy_from_list_of_tensor(test_dataTensor)).sum())

del test_tiles
del ret_split_test_data


### Glit predict

In [ ]:
full_predict = glue_data(result, split_data, shape_of_data)
print(full_predict.shape)

print("cut_first_class")
full_predict_first = full_predict[:,:,:,:1] if len(full_predict.shape) == 4 else full_predict
full_predict_first.shape

### Save predict model

In [ ]:
save_path_dir = f"model_predict/{UsemodelName}"
if not os.path.isdir(save_path_dir):
    print(f"Создаю {save_path_dir}")
    os.makedirs(save_path_dir)

saveDataframeAsImgs(save_path_dir, "LucchiPPtest", to_0_255_format_img(full_predict), classes_list[:number_of_classes])

## Calculete DICE and IoU

In [ ]:
test_etal_mito_frame = added_zero_channel_axis(test_mitoframe)
print(test_etal_mito_frame.shape)
print(test_etal_mito_frame.max())

In [ ]:
result_of_model = full_predict_first
print(result_of_model.shape)
print(result_of_model.max())

In [ ]:
binary_result = result_of_model.copy()
binary_result[binary_result < 0.5] = 0
binary_result[binary_result > 0] = 1

In [ ]:
print(f"Модель {UsemodelName} имеет качество по Dice {Dice(binary_result, test_etal_mito_frame)}")
print(f"Модель {UsemodelName} имеет качество по IoU {Jaccard(binary_result, test_etal_mito_frame)}")

In [ ]:
print (SYN_DATASET_PATH_MASK)

Можно попробовать запихать исходный через DivisiblePad

In [ ]:
print(device)

In [ ]:
del test_dataTensor
del full_predict_first
del full_predict

del syn_dataTensor
del syn_maskTensor

del dataTensor
del mitoTensor

del list_of_test_images
del test_dataframe
del list_of_test_masks
del test_mitoframe

del result

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
%whos

In [ ]:
torch.cuda.memory_allocated()

In [ ]:
print(torch.cuda.memory_summary())

# Загрузка кусочков многоклассового датасета

In [ ]:
TEST_DATASET_PATH_DATA = "D:/Data/datasets/Lucchi++/Test_In"

TEST_ETAL_MULTICLASS_PATH_DATA = "D:/Projects/UnetClass/pytorch3D/segmentation/data/original data/testing"
number_of_test_classes = number_of_classes
classes_etal_list = ["mitochondria", "boundaries", "vesicles", "axon", "PSD"]

# чтение всего стека датасета
print("\ntest_dataframe")
test_img_names_all = [name for name in os.listdir(TEST_DATASET_PATH_DATA) if name.endswith((AVAILABLE_IMG_TYPE))]
print(test_img_names_all)
list_of_test_images = [Image.open(os.path.join(TEST_DATASET_PATH_DATA, name)).convert("L") for name in test_img_names_all]
multi_test_dataframe = np.stack(list_of_test_images)
print(multi_test_dataframe.shape)
multi_test_dataframe = to_0_1_format_img(multi_test_dataframe)
print(multi_test_dataframe.min(), multi_test_dataframe.max())
multi_test_dataTensor = get_device_tensor_from_list_of_numpy(added_zero_channel_axis(multi_test_dataframe), device)
print(type(multi_test_dataTensor), multi_test_dataTensor.shape)

# чтение имеющейся разметки
print("\ntest_multimaskframe")

multi_test_img_names_all = [name for name in os.listdir(os.path.join(TEST_ETAL_MULTICLASS_PATH_DATA, classes_etal_list[0])) if name.endswith((AVAILABLE_IMG_TYPE))]
multi_test_img_names_all = sorted(multi_test_img_names_all, key=len)

print(check_sorted(multi_test_img_names_all))
print(multi_test_img_names_all)

print("\nполучение данных из масок ")
# получение данных из масок
test_multiclass_maskframe = []
for name in multi_test_img_names_all:
    one_mask_frame = []
    for i in range(number_of_test_classes):
        class_name = classes_etal_list[i]
        syn_mask = Image.open(os.path.join(TEST_ETAL_MULTICLASS_PATH_DATA, class_name, name)).convert("L")
        one_mask_frame.append(syn_mask)
    syn_mask = np.stack(one_mask_frame)
    syn_mask = to_0_1_format_img(syn_mask)
    print(syn_mask.shape, syn_mask.min(), syn_mask.max())
    test_multiclass_maskframe.append(syn_mask)

print(len(test_multiclass_maskframe))
test_multiclass_maskframe = np.transpose(np.stack(test_multiclass_maskframe), (0, 2, 3, 1))
print(test_multiclass_maskframe.shape)

mask_indexes_in_stak = []
for filename in multi_test_img_names_all:
    match = re.search(r'(\d+)', filename)
    if match:
        number_str = match.group(1)
        number = int(number_str)
        mask_indexes_in_stak.append(number)

print(mask_indexes_in_stak)


## Получение предикта

In [ ]:
multi_test_dataTensor.to("cpu")
multi_test_dataTensor.to(device)
pipliner.model.to("cpu")
pipliner.model.to(device)
print("Кручу-верчу джупитер обмануть хочу, чтобы он не гасил скорость предикта с 600 итерации (не полчается)")

In [ ]:
test_frames, split_data = split_3d_data(multi_test_dataTensor, shape_of_data, shape_of_data[0]-40)
result = predict(pipliner, data_list_to_predict_gen(test_frames), device)

## Склейка

In [ ]:
full_predict = glue_data(result, split_data, shape_of_data)
print(full_predict.shape)

In [ ]:
list_predicts_by_index = [full_predict[index] for index in mask_indexes_in_stak]
predict_multiframe = np.stack(list_predicts_by_index)
print(predict_multiframe.shape)

## Calculete DICE and IoU

In [ ]:
binary_multiresult = predict_multiframe.copy()
binary_multiresult[binary_multiresult < 0.5] = 0
binary_multiresult[binary_multiresult > 0] = 1

for n in range(number_of_test_classes):
    binary_result = binary_multiresult[:,:,:,n]
    etal_val = test_multiclass_maskframe[:,:,:,n]    

    print(f"Модель {UsemodelName} имеет качество по Dice для класса {classes_list[n]}: {Dice(binary_result, etal_val)}")
    print(f"Модель {UsemodelName} имеет качество по IoU  для класса {classes_list[n]}: {Jaccard(binary_result, etal_val)}")

In [ ]:
print (SYN_DATASET_PATH_MASK)

In [ ]:
del result
del multi_test_dataTensor
del full_predict